# DIP baseline reproduction on one image (Set5 baby, ×4) + visual comparison

Goal: get a baseline that matches a published number **before** re-running any ablation.

| Part | Runs | Target number | GPU time |
|---|---|---|---|
| A | **Authors' code** (capo-urjc/dip-sisr), plain L1, no regularizer, paper settings | Paper Table A5, baby L1 baseline **33.35 dB** | ~30 min |
| B | **Our code**, original DIP setup (Ulyanov 2017: MSE, Lanczos-2, PIL-Lanczos LR, 2000 it, lr 0.01, noise 0.03) | DIP paper, baby **31.49 dB** (Y channel, 4-px border) | ~10 min |
| C | **Our code** with the authors' pipeline (L1, bicubic LR, paper settings) | Must equal Part A if our copy of their pipeline is right | ~30 min |
| D | Visual comparison of any finished runs (full image + zoomed crop) | — | none |
| E | Summary table: ours vs. both papers | — | none |

Runtime: **GPU**. Secret `WANDB_API_KEY`. Run Setup, then any part; each part can be re-run on its own and is logged to W&B.

**Set5 has only HR images.** In every part the LR input is made from the HR image by a downsampler; Parts A and C use bicubic (`F.interpolate`, the authors' code), Part B uses PIL Lanczos (the original DIP code).

## Setup

In [ ]:
import os, sys, subprocess, importlib, json, time, shutil
from pathlib import Path
REPO_URL, BRANCH = "https://github.com/lakshjain7/deep-image-prior.git", "claude/epic-ride-rnqbk4"
REPO_DIR = Path("/content/deep-image-prior")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "origin", BRANCH], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, str(REPO_DIR))
for mod, pkg in [("wandb", "wandb"), ("skimage", "scikit-image"), ("cv2", "opencv-python-headless"), ("torchmetrics", "torchmetrics")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch, wandb, numpy as np, pandas as pd
from PIL import Image
from IPython.display import display
try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass
wandb.login()
import stage3
from evaluate_set5 import calculate_psnr_y
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

PROJECT = "dip-sr-regularizers"
tracker = stage3.Tracker(None, PROJECT, Path("/content/repro_runs"))
api = wandb.Api()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE", "| W&B entity:", tracker.entity)

# Set5 HR images
SET5_DIR = Path("/content/Set5")
try:
    image_paths = stage3.find_set5(SET5_DIR)
except FileNotFoundError:
    import urllib.request, tarfile
    SET5_DIR.mkdir(parents=True, exist_ok=True)
    tmp = SET5_DIR / "Set5_HR.tar.gz"
    urllib.request.urlretrieve("https://huggingface.co/datasets/eugenesiow/Set5/resolve/main/data/Set5_HR.tar.gz", tmp)
    tarfile.open(tmp).extractall(SET5_DIR)
    image_paths = stage3.find_set5(SET5_DIR)
IMAGE = "baby"
HR = np.asarray(Image.open(image_paths[IMAGE]).convert("RGB"), dtype=np.float32) / 255
print(IMAGE, HR.shape)

def scores(sr, hr=HR):
    """RGB PSNR/SSIM on the whole image + Y-channel PSNR with a 4-px border (DIP-paper protocol). HWC float in [0,1]."""
    sr = np.clip(sr, 0, 1).astype(np.float32)
    return {"psnr_rgb": psnr(hr, sr, data_range=1.0), "ssim_rgb": ssim(hr, sr, channel_axis=2, data_range=1.0),
            "psnr_y_border4": calculate_psnr_y(hr[4:-4, 4:-4], sr[4:-4, 4:-4])}

## A. Authors' code: plain L1 baseline (paper settings)

The published `main.py` does not run unchanged. The cell applies **only** these fixes, prints each, and changes nothing else:

1. `datasets/` has no `__init__.py`, so Colab's Hugging Face `datasets` package hides it → `ModuleNotFoundError: datasets.NaturalColor` (the error from our first notebook). Fix: add an empty `__init__.py`.
2. `CUDA_VISIBLE_DEVICES = "3"` is hard-coded (their lab machine); Colab has one GPU (index 0). Fix: comment it out.
3. `--factor` is parsed as a float (4.0); their Lanczos kernel code then crashes in NumPy (`'float' object cannot be interpreted as an integer`). Fix: pass `int(args.factor)`.
4. Their script does not save the output image (the call is commented out). Fix: save it next to the results, for the visual comparison. Metrics are unaffected.

Settings = `main.py` defaults except where the paper's baseline needs otherwise: `--method 0` (L1, single Lanczos-2), every regularizer weight 0 (the default turns TVSSAH **on** at 0.8), `--factor 4`, 5500 iterations, lr 0.001, noise 0.01, seed 42 (fixed in their code).

In [ ]:
AUTHORS_COMMIT = "e2c80f0e60880a868e77dd024b70cb4a2e4b3ed5"
FACTOR = 4                       # paper tables are ×4 by all appearances; set 2 to test the code's default
REF = Path("/content/dip-sisr")
if not REF.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/capo-urjc/dip-sisr.git", str(REF)], check=True)
subprocess.run(["git", "-C", str(REF), "checkout", "-q", "--", "."], check=True)     # undo earlier patches
subprocess.run(["git", "-C", str(REF), "checkout", "-q", AUTHORS_COMMIT], check=True)

main_py = REF / "main.py"; src = main_py.read_text()
patches = [
    ('os.environ["CUDA_VISIBLE_DEVICES"] = "3"', '# os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # patched: Colab has one GPU'),
    ("factor=args.factor,", "factor=int(args.factor),  # patched: float factor crashes the kernel code"),
    ("        qm_mae, qm_mse, qm_ssim, qm_psnr, qm_sam = compute_qlt_measures(infered=full_sr, gt=gt)\n",
     "        qm_mae, qm_mse, qm_ssim, qm_psnr, qm_sam = compute_qlt_measures(infered=full_sr, gt=gt)\n"
     "        torch.save(full_sr.detach().cpu(), f'sr_output_{i}.pt')  # patched: keep the output for comparison\n"),
]
for old, new in patches:
    assert src.count(old) == 1, f"patch target not found exactly once: {old[:60]}"
    src = src.replace(old, new); print("patched:", new.strip()[:110])
main_py.write_text(src)
(REF / "datasets" / "__init__.py").write_text(""); print("patched: added datasets/__init__.py")

data_dir = REF / "DATA" / "Set5"
shutil.rmtree(data_dir, ignore_errors=True); data_dir.mkdir(parents=True)
shutil.copy(image_paths[IMAGE], data_dir / f"{IMAGE}.png")          # one image only
for f in [REF / "results_experiments" / "Set5.csv", REF / "sr_output_0.pt"]:
    if f.exists(): f.unlink()

cmd = [sys.executable, "main.py", "--method", "0", "--n_its", "5500", "--lr", "0.001", "--reg_noise_std", "0.01",
       "--dataset", "Set5", "--input_depth", "32", "--tv_param", "0", "--nn_param", "0", "--ssahtv_param", "0",
       "--schiavi_param", "0", "--factor", str(FACTOR)]
print("running:", " ".join(cmd[1:]))
t0 = time.time()
with open("/content/authors_run.log", "w") as log:
    rc = subprocess.run(cmd, cwd=REF, stdout=log, stderr=subprocess.STDOUT).returncode
print(f"exit code {rc} after {(time.time() - t0) / 60:.1f} min")
print(open("/content/authors_run.log").read()[-1500:])
assert rc == 0, "authors' run failed - see the log above"

row = pd.read_csv(REF / "results_experiments" / "Set5.csv").iloc[-1]
sr_a = torch.load(REF / "sr_output_0.pt")[0].permute(1, 2, 0).numpy()
A = {"their_psnr (torchmetrics)": float(row.qm_psnr), "their_ssim": float(row.qm_ssim),
     "their_best_psnr_during_training": float(row.best_psnr), **scores(sr_a)}
Image.fromarray((np.clip(sr_a, 0, 1) * 255).round().astype(np.uint8)).save(f"/content/authors_{IMAGE}_x{FACTOR}.png")
print(json.dumps({k: round(v, 3) for k, v in A.items()}, indent=1))
print("Paper Table A5, baby, L1 baseline: 33.35 dB")

with wandb.init(project=PROJECT, job_type="repro-authors-code", name=f"repro_authors_{IMAGE}_l1_x{FACTOR}",
                tags=["reproduction", "authors-code"], config={"image": IMAGE, "factor": FACTOR, "code": "capo-urjc/dip-sisr",
                "commit": AUTHORS_COMMIT, "cmd": " ".join(cmd[1:]), "patches": [p[1] for p in patches] + ["datasets/__init__.py"]}) as run:
    run.summary.update(A)
    run.log({"sr": wandb.Image(f"/content/authors_{IMAGE}_x{FACTOR}.png")})

## B. Our code: original DIP setup (Ulyanov et al. 2017)

MSE loss, Lanczos-2 forward model, LR made by PIL Lanczos (as in the original DIP notebook), 2000 iterations, lr 0.01, noise 0.03, seed 7. The DIP paper reports baby ×4 = **31.49 dB**, measured on the Y channel with a 4-pixel border (our `psnr_y_border4`).

In [ ]:
W = dict(stage3.DEFAULT_REG_WEIGHTS)
def run_ours(loss, reg, protocol, lr_method):
    job = stage3.build_jobs([IMAGE], [loss], [reg], [7], protocol, W, lr_method)[0]
    P = stage3.PROTOCOLS[protocol]
    cfg = {"factor": 4, "lr": P["lr"], "reg_noise_std": P["reg_noise_std"], "num_iter": P["num_iter"], "log_freq": 100,
           "protocol_note": P["note"], "tags": ["reproduction", protocol, lr_method], "job_type": "repro-ours"}
    rec = stage3.run_job(job, cfg, image_paths[IMAGE], tracker, "/content/repro_logs")
    print(rec["status"], job["key"], f"({rec.get('elapsed_s', 0) / 60:.1f} min)" if "elapsed_s" in rec else "")
    if rec["status"] == "failed": print(rec.get("error_tail", ""))
    return job["key"]

key_b = run_ours("mse", "none", "adapted", "pil-lanczos")

## C. Our code with the authors' pipeline

L1, no regularizer, bicubic LR via `F.interpolate` (no antialiasing), paper settings. If our copy of their pipeline is faithful, this should land close to Part A (different random seeds, so small differences are expected).

In [ ]:
key_c = run_ours("l1", "none", "paper", "bicubic")

## D. Visual comparison

Shows any finished runs side by side: the full image and a zoomed crop, each titled with its PSNR against the real image. Add run keys to `RUNS` (keys are listed in the results notebook) to compare other configurations. Images come from each run's W&B artifact.

In [ ]:
import matplotlib.pyplot as plt

def run_image(key):
    ok = [r for r in tracker._query(key) if r.summary.get("status") == "complete"]
    if not ok: return None
    r = max(ok, key=lambda r: r.created_at)
    path = Path(api.artifact(f"{tracker.entity}/{PROJECT}/result-{r.id}:latest").download(root=f"/content/art/{r.id}"))
    return np.asarray(Image.open(next(path.glob("final_sr_x*.png"))).convert("RGB"), dtype=np.float32) / 255

def bicubic_up(hr, lr_method):
    t = torch.from_numpy(hr).permute(2, 0, 1)[None]
    if lr_method == "pil-lanczos":
        im = Image.fromarray((hr * 255).round().astype(np.uint8)); w, h = im.size
        lr = im.resize((w // 4, h // 4), Image.Resampling.LANCZOS)
    else:
        lr_t = torch.nn.functional.interpolate(t, scale_factor=0.25, mode="bicubic", align_corners=False,
                                               antialias=(lr_method == "bicubic-aa"))
        lr = Image.fromarray((lr_t[0].permute(1, 2, 0).clamp(0, 1).numpy() * 255).round().astype(np.uint8))
    return np.asarray(lr.resize((hr.shape[1], hr.shape[0]), Image.BICUBIC), dtype=np.float32) / 255

RUNS = [  # (label, run key) - edit freely
    ("DIP original (MSE, Lanczos LR)", f"{IMAGE}__mse__none-0__seed7__adapted"),
    ("L1, clean LR", f"{IMAGE}__l1__none-0__seed7__adapted"),
    ("L1 + TV, clean LR", f"{IMAGE}__l1__tv-0.5__seed7__adapted"),
    ("L1, aliased LR", f"{IMAGE}__l1__none-0__seed7__adapted__bicubic"),
    ("L1 + TV, aliased LR", f"{IMAGE}__l1__tv-0.5__seed7__adapted__bicubic"),
    ("Our copy of authors' pipeline", f"{IMAGE}__l1__none-0__seed7__paper__bicubic"),
]
CROP = (slice(170, 298), slice(140, 268))   # rows, cols of the zoom (baby's eye); change for other images

panels = [("Real image (HR)", HR), ("Bicubic upsampling of clean LR", bicubic_up(HR, "pil-lanczos")),
          ("Bicubic upsampling of aliased LR", bicubic_up(HR, "bicubic"))]
authors_png = Path(f"/content/authors_{IMAGE}_x4.png")
if authors_png.exists():
    panels.append(("Authors' code (Part A)", np.asarray(Image.open(authors_png), dtype=np.float32) / 255))
for label, key in RUNS:
    im = run_image(key)
    if im is None: print("not found / not finished:", key)
    else: panels.append((label, im))

fig, axes = plt.subplots(2, len(panels), figsize=(3.2 * len(panels), 6.6))
for j, (label, im) in enumerate(panels):
    s = scores(im)
    axes[0, j].imshow(np.clip(im, 0, 1)); axes[1, j].imshow(np.clip(im[CROP], 0, 1))
    axes[0, j].set_title(f"{label}\nPSNR {s['psnr_rgb']:.2f} dB" if label != "Real image (HR)" else label, fontsize=9)
    for a in axes[:, j]: a.axis("off")
plt.tight_layout(); plt.savefig(f"/content/visual_comparison_{IMAGE}.png", dpi=130); plt.show()
with wandb.init(project=PROJECT, job_type="visual-comparison", name=f"visual_{IMAGE}", tags=["visual"]) as run:
    run.log({"comparison": wandb.Image(f"/content/visual_comparison_{IMAGE}.png")})
print("saved /content/visual_comparison_" + IMAGE + ".png (also logged to W&B)")

## E. Summary: ours vs. the two papers (baby ×4)

In [ ]:
rows = []
if "A" in globals():
    rows.append(dict(run="A. Authors' code, L1, paper settings", **A, target="33.35 (paper Table A5, RGB)"))
for label, key, target in [("B. Our code, original DIP (MSE)", f"{IMAGE}__mse__none-0__seed7__adapted", "31.49 (DIP paper, Y + border)"),
                           ("C. Our code, authors' pipeline", f"{IMAGE}__l1__none-0__seed7__paper__bicubic", "= Part A"),
                           ("Earlier: our L1, clean LR", f"{IMAGE}__l1__none-0__seed7__adapted", "-")]:
    im = run_image(key)
    if im is not None: rows.append(dict(run=label, **scores(im), target=target))
display(pd.DataFrame(rows).set_index("run").round(3))

### How to read Part E

- **A ≈ 33.35:** their pipeline is reproduced; compare C with A to see whether our copy matches.
- **A well below 33.35:** even their own code does not give the published number with the stated settings. Try `FACTOR = 2` (the code's default) and/or ask the authors which settings produced Table A5.
- **B ≈ 31.49 (Y column):** the original DIP is reproduced with our code; that is the baseline to re-run the ablations from.
- **C ≠ A by more than ~0.5 dB:** our copy of their pipeline differs somewhere (cropping, metric, seed); inspect before ablating.